# microWakeWord Training Notebook

Train a custom wake word detection model compatible with ESP32-S3.

## Overview

This notebook trains models compatible with `wakeword_pretrained_test.ino`:
- **Input**: 40-bin log-mel spectrogram features (30ms window, 10ms stride)
- **Architecture**: MixNet-based streaming model
- **Output**: Quantized TFLite model for ESP32

Based on: https://github.com/kahrendt/microWakeWord

In [38]:
# Cell 1: Install dependencies

# Install audio-metadata fork (required)
!uv add 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

# Install pymicro-features fork with correct API (ProcessSamples method)
!uv add 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# Clone and install microWakeWord
!git clone https://github.com/kahrendt/microWakeWord
!uv add ./microWakeWord

print("microWakeWord installed!")

Resolved 268 packages in 999ms                                       
Audited 259 packages in 7ms
Resolved 268 packages in 674ms                                       
Audited 259 packages in 7ms
fatal: destination path 'microWakeWord' already exists and is not an empty directory.
Resolved 268 packages in 1.44s                                       
Prepared 1 package in 197ms                                              
Uninstalled 1 package in 1ms
Installed 1 package in 2ms0 (from file:///workspace/microWak
 ~ microwakeword==0.1.0 (from file:///workspace/microWakeWord)
microWakeWord installed!


In [39]:
# Cell 2: Install TTS engines (Piper + XTTS-v2)
!git clone https://github.com/rhasspy/piper-sample-generator

# Download Piper TTS model
!mkdir -p piper-sample-generator/models
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
    'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

fatal: destination path 'piper-sample-generator' already exists and is not an empty directory.


In [21]:
# Install PyTorch and XTTS-v2
!uv add torch==2.5 torchdata torchvision==0.20 torchaudio==2.5 torchcodec==0.1 datasets==3.6.0 soundfile numba librosa TTS ipywidgets coqui-tts piper-tts  'tensorflow==2.17.*'

print("Piper TTS + XTTS-v2 installed!")

Resolved 268 packages in 9.01s                                       
Prepared 3 packages in 29.51s                                            
Uninstalled 3 packages in 1.22s
Installed 3 packages in 8.32s                               
 - protobuf==5.29.5
 + protobuf==4.25.8
 - tensorboard==2.18.0
 + tensorboard==2.17.1
 - tensorflow==2.18.1
 + tensorflow==2.17.1
Piper TTS + XTTS-v2 installed!


In [23]:
# Patch microWakeWord for TensorFlow 2.17 compatibility
# TF 2.17 returns numpy arrays directly instead of tensors, so .numpy() calls fail

import re

train_py = '/workspace/microWakeWord/microwakeword/train.py'

with open(train_py, 'r') as f:
    content = f.read()

# Check if already patched
if 'np.asarray' not in content:
    # Replace patterns like: result["fp"].numpy() -> np.asarray(result["fp"])
    # This handles both tensor (needs .numpy()) and numpy array (already is) cases
    patched = re.sub(
        r'(\w+)\[(["\'])(\w+)\2\]\.numpy\(\)',
        r'np.asarray(\1["\3"])',
        content
    )
    
    with open(train_py, 'w') as f:
        f.write(patched)
    
    print("✓ Patched train.py for TensorFlow 2.17 compatibility")
else:
    print("✓ train.py already patched")

✓ Patched train.py for TensorFlow 2.17 compatibility


In [1]:
# Cell 3: Setup directories
import os
import json
from pathlib import Path

WORK_DIR = Path('/workspace')
WORK_DIR.mkdir(parents=True, exist_ok=True)

GENERATED_DIR = WORK_DIR / 'generated_samples'
FEATURES_DIR = WORK_DIR / 'features'
AUGMENTATION_DIR = WORK_DIR / 'augmentation'
MODEL_DIR = WORK_DIR / 'trained_models'  # DON'T create - training script creates it

# Only create directories that we manage, not MODEL_DIR
for d in [GENERATED_DIR, FEATURES_DIR, AUGMENTATION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {WORK_DIR}")

Working directory: /workspace


## Step 1: Define Your Wake Word

In [2]:
# Cell 4: Configure wake word - "Hey Daisy" (EXPANDED)
WAKE_WORD = "Hey Daisy"
WAKE_WORD_ID = "hey_daisy"

# EXPANDED phonetic variations for diverse TTS samples
PHONETIC_VARIATIONS = [
    # Standard variations
    "Hey Daisy ",
    "hey Daisy ", 
    "hey daisy ",
    "Hey Daisy. ",
    "Hey Daisy! ",
    # Phonetic spellings
    "hey day zee ",
    "hey day-zee ",
    "hey daisee ",
    "hey daysy ",
    "hey daezy ",
    "hay daisy ",
    "hay Daisy ",
    # Speed/emphasis variations
    "HEY Daisy ",
    "hey DAISY ",
    "HEY DAISY ",
    # Casual/natural variations
    "hey daisy? ",
    "heyyy daisy ",
    "hey daizy ",
    "hey dayzie ",
    "hey dazey ",
]

# Write variations to file for Piper
variations_file = WORK_DIR / 'variations.txt'
with open(variations_file, 'w') as f:
    for v in PHONETIC_VARIATIONS:
        f.write(f"{v}\n")

print(f"Wake word: {WAKE_WORD}")
print(f"Variations: {len(PHONETIC_VARIATIONS)} (was 12, now {len(PHONETIC_VARIATIONS)})")

Wake word: Hey Daisy
Variations: 20 (was 12, now 20)


## Step 2: Generate Synthetic Samples

In [3]:
# Cell 5: Generation config (SKIP IF DATA EXISTS)
import subprocess
import shutil
import gc
from pathlib import Path

training_dir = GENERATED_DIR / 'training'
training_dir.mkdir(exist_ok=True)

#=============================================================================
# GENERATION CONFIG
#=============================================================================
TED_SPEAKERS = ['BillGates', 'DaphneKoller', 'FeiFeiLi', 'GeorgeTakei', 
                'JaneGoodall', 'SalmanKhan', 'StephenHawking', 'StephenWolfram']
SAMPLES_PER_SPEAKER = 10
PIPER_SAMPLES_PER_VAR = 100

total_voices = len(TED_SPEAKERS) * SAMPLES_PER_SPEAKER  # 80
total_piper = len(PHONETIC_VARIATIONS) * PIPER_SAMPLES_PER_VAR
total_xtts = len(PHONETIC_VARIATIONS) * total_voices

# Check existing samples
existing_piper = len(list(training_dir.glob('piper_*.wav')))
existing_xtts = len(list(training_dir.glob('xtts_*.wav')))
existing_total = existing_piper + existing_xtts

print(f"Target: {total_piper + total_xtts} samples")
print(f"  Piper: {total_piper} ({PIPER_SAMPLES_PER_VAR}/variation)")
print(f"  XTTS:  {total_xtts} ({total_voices} voices/variation)")
print()

if existing_total > 0:
    print(f"✓ EXISTING DATA FOUND: {existing_total} samples")
    print(f"  Piper: {existing_piper}")
    print(f"  XTTS:  {existing_xtts}")
    print("  (Generation cells will be skipped)")
    SKIP_GENERATION = True
    piper_count = existing_piper
    xtts_count = existing_xtts
else:
    print("No existing data, will generate fresh samples")
    SKIP_GENERATION = False

Target: 3600 samples
  Piper: 2000 (100/variation)
  XTTS:  1600 (80 voices/variation)

✓ EXISTING DATA FOUND: 3600 samples
  Piper: 2000
  XTTS:  1600
  (Generation cells will be skipped)


In [4]:
#=============================================================================
# PHASE 1: PIPER GENERATION (SKIP IF DATA EXISTS)
#=============================================================================
if SKIP_GENERATION:
    print("✓ Skipping Piper generation - using existing data")
else:
    print("="*60)
    print("PHASE 1: PIPER TTS")
    print("="*60)

    for i, variation in enumerate(PHONETIC_VARIATIONS):
        print(f"  [{i+1}/{len(PHONETIC_VARIATIONS)}] '{variation}'")
        temp_dir = GENERATED_DIR / f'temp_piper_{i}'
        temp_dir.mkdir(exist_ok=True)
        
        subprocess.run([
            'python3', 'piper-sample-generator/generate_samples.py',
            variation,
            '--max-samples', str(PIPER_SAMPLES_PER_VAR),
            '--batch-size', '10',
            '--output-dir', str(temp_dir),
            '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt'
        ], capture_output=True)
        
        for wav in temp_dir.glob('*.wav'):
            new_name = f"piper_var{i}_{wav.name}"
            shutil.move(str(wav), str(training_dir / new_name))
        shutil.rmtree(temp_dir, ignore_errors=True)

    piper_count = len(list(training_dir.glob('piper_*.wav')))
    print(f"\n✓ Piper complete: {piper_count} samples")
    gc.collect()

✓ Skipping Piper generation - using existing data


In [5]:
#=============================================================================
# PHASE 2: DOWNLOAD TED SPEAKER SAMPLES
#=============================================================================
print("\n" + "="*60)
print("PHASE 2: DOWNLOAD TED SPEAKERS")
print("="*60)

import urllib.request
import librosa
import soundfile as sf

speaker_voices_dir = WORK_DIR / 'speaker_voices'
speaker_voices_dir.mkdir(exist_ok=True)

speaker_files = []
for speaker in TED_SPEAKERS:
    for sample_idx in range(SAMPLES_PER_SPEAKER):
        wav_path = speaker_voices_dir / f'ted_{speaker}_{sample_idx}.wav'
        mp3_path = speaker_voices_dir / f'ted_{speaker}_{sample_idx}.mp3'
        
        if not wav_path.exists():
            try:
                url = f'https://raw.githubusercontent.com/audio-samples/audio-samples.github.io/master/samples/mp3/ted_speakers/{speaker}/sample-{sample_idx}.mp3'
                urllib.request.urlretrieve(url, mp3_path)
                audio, sr = librosa.load(mp3_path, sr=16000, mono=True)
                if len(audio) > 6 * 16000:
                    audio = audio[:6 * 16000]
                sf.write(str(wav_path), audio, 16000)
                mp3_path.unlink()
            except Exception as e:
                print(f"  ✗ {speaker}/sample-{sample_idx}: {e}")
                continue
        speaker_files.append(str(wav_path))
    print(f"  ✓ {speaker} ({SAMPLES_PER_SPEAKER} samples)")

print(f"\n✓ Downloaded {len(speaker_files)} voice references")


PHASE 2: DOWNLOAD TED SPEAKERS
  ✓ BillGates (10 samples)
  ✓ DaphneKoller (10 samples)
  ✓ FeiFeiLi (10 samples)
  ✓ GeorgeTakei (10 samples)
  ✓ JaneGoodall (10 samples)
  ✓ SalmanKhan (10 samples)
  ✓ StephenHawking (10 samples)
  ✓ StephenWolfram (10 samples)

✓ Downloaded 80 voice references


In [6]:
#=============================================================================
# PHASE 3: XTTS GENERATION (SKIP IF DATA EXISTS)
#=============================================================================
if SKIP_GENERATION:
    print("✓ Skipping XTTS generation - using existing data")
else:
    print("\n" + "="*60)
    print("PHASE 3: XTTS-v2 GENERATION")
    print("="*60)

    import torch
    from TTS.api import TTS

    # Build task list: (variation_text, speaker_wav, output_path)
    xtts_tasks = []
    for i, variation in enumerate(PHONETIC_VARIATIONS):
        for speaker_wav in speaker_files:
            speaker_name = Path(speaker_wav).stem
            output_path = str(training_dir / f"xtts_var{i}_{speaker_name}.wav")
            # Skip if already generated (resume support)
            if not Path(output_path).exists():
                xtts_tasks.append((variation, speaker_wav, output_path))

    total_tasks = len(PHONETIC_VARIATIONS) * len(speaker_files)
    already_done = total_tasks - len(xtts_tasks)

    print(f"  Total XTTS tasks: {total_tasks}")
    if already_done > 0:
        print(f"  Already completed: {already_done} (resuming...)")
    print(f"  Remaining: {len(xtts_tasks)}")

    if len(xtts_tasks) > 0:
        # Load XTTS model
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"\n  Loading XTTS-v2 on {device}...")
        xtts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
        print("  Model loaded!")
        
        # Sequential generation with progress
        failed = 0
        for idx, (variation, speaker_wav, output_path) in enumerate(xtts_tasks):
            # Progress every 80 samples (1 variation * 80 voices)
            if idx % 80 == 0:
                pct = 100 * (idx + already_done) // total_tasks
                print(f"  Progress: {idx + already_done}/{total_tasks} ({pct}%)")
            
            try:
                xtts.tts_to_file(
                    text=variation,
                    file_path=output_path,
                    speaker_wav=speaker_wav,
                    language="en"
                )
            except Exception as e:
                failed += 1
                if failed <= 5:  # Only show first 5 errors
                    print(f"    ✗ {Path(output_path).stem}: {e}")
        
        # Cleanup
        del xtts
        gc.collect()
        torch.cuda.empty_cache()
        
        if failed > 0:
            print(f"\n  ⚠ {failed} samples failed")

    xtts_count = len(list(training_dir.glob('xtts_*.wav')))
    print(f"\n✓ XTTS complete: {xtts_count} samples")
    gc.collect()

✓ Skipping XTTS generation - using existing data


In [7]:
#=============================================================================
# SUMMARY
#=============================================================================
print("\n" + "="*60)
total_count = len(list(training_dir.glob('*.wav')))
print(f"✓ GENERATION COMPLETE: {total_count} total samples")
print(f"  Piper: {piper_count}")
print(f"  XTTS:  {xtts_count}")
print("="*60)


✓ GENERATION COMPLETE: 3600 total samples
  Piper: 2000
  XTTS:  1600


## Step 3: Download Augmentation Data

In [8]:
# Cell 7: Download MIT impulse responses
import urllib.request
import zipfile

MIT_IR_DIR = AUGMENTATION_DIR / 'mit_ir'
MIT_IR_DIR.mkdir(exist_ok=True)
ir_zip = MIT_IR_DIR / 'Audio.zip'

print("Downloading MIT impulse responses...")
if not ir_zip.exists():
    urllib.request.urlretrieve(
        "https://mcdermottlab.mit.edu/Reverb/IRMAudio/Audio.zip", ir_zip)
    with zipfile.ZipFile(ir_zip, 'r') as z:
        z.extractall(MIT_IR_DIR)
print("Done!")

Done!


In [9]:
# Cell 8: Download background noise audio
# Install torchcodec FIRST (required for datasets audio decoding)

import subprocess
from pathlib import Path

AUDIOSET_DIR = AUGMENTATION_DIR / 'audioset'
AUDIOSET_DIR.mkdir(exist_ok=True)

# Check if we already have audio files
existing = list(AUDIOSET_DIR.glob('*.wav'))
if len(existing) >= 100:
    print(f"AudioSet already has {len(existing)} samples, skipping download")
else:
    try:
        from datasets import load_dataset
        import soundfile as sf
        print("Loading AudioSet subset via datasets library...")
        
        # Load balanced subset with streaming
        ds = load_dataset(
            "agkphysics/AudioSet", 
            split="train",
            streaming=True,
        )
        
        # Save first 500 audio samples as wav files
        count = 0
        max_samples = 500
        
        for sample in ds:
            if count >= max_samples:
                break
            try:
                audio = sample['audio']
                wav_path = AUDIOSET_DIR / f"audioset_{count:04d}.wav"
                sf.write(str(wav_path), audio['array'], audio['sampling_rate'])
                count += 1
                if count % 100 == 0:
                    print(f"  Saved {count}/{max_samples} samples")
            except Exception:
                continue  # Skip problematic samples
        
        print(f"Done! Saved {count} background audio samples")

    except Exception as e:
        print(f"AudioSet loading failed: {e}")
        print("Background augmentation will use MIT impulse responses only.")

Loading AudioSet subset via datasets library...


Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

  Saved 100/500 samples
  Saved 200/500 samples
  Saved 300/500 samples
  Saved 400/500 samples
  Saved 500/500 samples
Done! Saved 500 background audio samples


In [10]:
# Cell 9: Download pre-computed negative features from HuggingFace
import zipfile

NEGATIVE_DIR = FEATURES_DIR / 'negatives'
NEGATIVE_DIR.mkdir(exist_ok=True)

# These are pre-computed spectrogram feature files (not raw audio)
NEGATIVE_FILES = {
    'speech.zip': 'speech',
    'no_speech.zip': 'no_speech',
    'dinner_party.zip': 'dinner_party',
    'dinner_party_eval.zip': 'dinner_party_eval',
}

BASE_URL = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main'

print("Downloading negative features (this may take several minutes)...")
for zip_name, folder_name in NEGATIVE_FILES.items():
    zip_path = NEGATIVE_DIR / zip_name
    extract_dir = NEGATIVE_DIR / folder_name
    
    if not extract_dir.exists():
        print(f"  Downloading {zip_name}...")
        !wget -q --show-progress -O {zip_path} '{BASE_URL}/{zip_name}'
        
        if zip_path.exists() and zip_path.stat().st_size > 1000:
            print(f"  Extracting {zip_name}...")
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(NEGATIVE_DIR)
            zip_path.unlink()  # Remove zip to save space
        else:
            print(f"  WARNING: {zip_name} download failed")
    else:
        print(f"  {folder_name} already exists")

print("Done!")

  speech already exists
  no_speech already exists
  dinner_party already exists
  dinner_party_eval already exists
Done!


## Step 4: Generate Spectrogram Features

In [11]:
# Cell 10: Configure augmentation
from microwakeword.audio.augmentation import Augmentation

# Verify we have background noise files
audioset_files = list(AUDIOSET_DIR.glob('*.wav'))
if len(audioset_files) == 0:
    raise RuntimeError(f"No AudioSet files found in {AUDIOSET_DIR}! Re-run AudioSet download cell.")

augmentation = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.1,
        "TanhDistortion": 0.1,
        "PitchShift": 0.1,
        "BandStopFilter": 0.1,
        "AddColorNoise": 0.1,
        "AddBackgroundNoise": 0.75,
        "Gain": 1.0,
        "RIR": 0.5,
    },
    impulse_paths=[str(MIT_IR_DIR / 'Audio')],
    background_paths=[str(AUDIOSET_DIR)],
)
print(f"Augmentation configured with {len(audioset_files)} background noise files")

Augmentation configured with 500 background noise files


In [14]:
# Cell 11: Generate spectrogram features from audio samples
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap
import os

POSITIVE_FEATURES_DIR = FEATURES_DIR / 'positive'

# Check if features already exist
train_mmap = POSITIVE_FEATURES_DIR / 'training' / 'wakeword_mmap'
if train_mmap.exists():
    print("✓ Features already exist, skipping generation")
else:
    print("Generating spectrogram features from audio samples...")
    print(f"  Source: {training_dir} ({len(list(training_dir.glob('*.wav')))} wav files)")
    print(f"  Output: {POSITIVE_FEATURES_DIR}")
    print()
    
    # Create Clips object pointing to audio files
    clips = Clips(
        input_directory=str(training_dir),
        file_pattern='*.wav',
        max_clip_duration_s=None,
        remove_silence=False,
        random_split_seed=10,
        split_count=0.1,  # 10% for validation
    )
    
    # Create output directories
    POSITIVE_FEATURES_DIR.mkdir(parents=True, exist_ok=True)
    
    splits = ["training", "validation"]
    for split in splits:
        out_dir = POSITIVE_FEATURES_DIR / split
        out_dir.mkdir(exist_ok=True)
        
        split_name = "train" if split == "training" else "validation"
        repetition = 10 if split == "training" else 1  # More augmented copies for training
        
        spectrograms = SpectrogramGeneration(
            clips=clips,
            augmenter=augmentation,
            slide_frames=10,
            step_ms=10,
        )
        
        print(f"  Generating {split} features (repeat={repetition})...")
        RaggedMmap.from_generator(
            out_dir=str(out_dir / 'wakeword_mmap'),
            sample_generator=spectrograms.spectrogram_generator(
                split=split_name, 
                repeat=repetition
            ),
            batch_size=100,
            verbose=True,
        )
    
    print("\n✓ Feature generation complete")

# Verify
train_features = RaggedMmap(str(POSITIVE_FEATURES_DIR / 'training' / 'wakeword_mmap'), mode='r')
val_features = RaggedMmap(str(POSITIVE_FEATURES_DIR / 'validation' / 'wakeword_mmap'), mode='r')
print(f"  Training features: {len(train_features)} spectrograms")
print(f"  Validation features: {len(val_features)} spectrograms")

2025-11-26 13:01:28.083195: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764162088.134152   29392 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764162088.148085   29392 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Generating spectrogram features from audio samples...
  Source: /workspace/generated_samples/training (3600 wav files)
  Output: /workspace/features/positive

  Generating training features (repeat=10)...


0it [00:00, ?it/s]

  Generating validation features (repeat=1)...


0it [00:00, ?it/s]


✓ Feature generation complete
  Training features: 288000 spectrograms
  Validation features: 3600 spectrograms


In [15]:
# Cell 12: Create training config (SCALED UP + GPU OPTIMIZED)
import yaml
import os

#=============================================================================
# GPU OPTIMIZATION CONFIG - ADJUST FOR YOUR HARDWARE
#=============================================================================
# vast.ai 45GB+ VRAM: batch_size=512 or 1024
# Colab T4 (15GB): batch_size=128
# Colab free (12GB RAM): batch_size=64
BATCH_SIZE = 1024

# Enable TensorFlow optimizations
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # Reduce log spam

print(f"Training config:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Training steps: 20,000")
print()

# Verify features exist
print("Checking feature directories...")
print(f"  Positive features: {len(train_features)} training, {len(val_features)} validation")

# Training config with larger batch size
training_config = {
    'train_dir': str(MODEL_DIR),
    'window_step_ms': 10,
    'clip_duration_ms': 1500,
    'batch_size': BATCH_SIZE,
    
    # Training schedule
    'training_steps': [20000],
    'learning_rates': [0.001],
    
    # Class weights
    'positive_class_weight': [1],
    'negative_class_weight': [20],
    
    # SpecAugment parameters
    'time_mask_max_size': [10],
    'time_mask_count': [2],
    'freq_mask_max_size': [3],
    'freq_mask_count': [2],
    
    # Evaluation
    'eval_step_interval': 1000,
    'target_minimization': 0.5,
    'minimization_metric': 'loss',
    'maximization_metric': 'accuracy',
    
    # Feature datasets
    'features': [
        {
            'type': 'mmap',
            'features_dir': str(POSITIVE_FEATURES_DIR),
            'truth': True,
            'sampling_weight': 2.0,
            'penalty_weight': 1.0,
            'truncation_strategy': 'truncate_start',
        },
        {
            'type': 'mmap',
            'features_dir': str(NEGATIVE_DIR / 'speech'),
            'truth': False,
            'sampling_weight': 10.0,
            'penalty_weight': 1.0,
            'truncation_strategy': 'random',
        },
        {
            'type': 'mmap',
            'features_dir': str(NEGATIVE_DIR / 'dinner_party'),
            'truth': False,
            'sampling_weight': 10.0,
            'penalty_weight': 1.0,
            'truncation_strategy': 'random',
        },
        {
            'type': 'mmap',
            'features_dir': str(NEGATIVE_DIR / 'no_speech'),
            'truth': False,
            'sampling_weight': 5.0,
            'penalty_weight': 1.0,
            'truncation_strategy': 'random',
        },
    ],
}

config_path = WORK_DIR / 'training_config.yaml'
with open(config_path, 'w') as f:
    yaml.dump(training_config, f, default_flow_style=False)

print(f"\n✓ Config saved: {config_path}")

Training config:
  Batch size: 1024
  Training steps: 20,000

Checking feature directories...
  Positive features: 288000 training, 3600 validation

✓ Config saved: /workspace/training_config.yaml


## Step 5: Configure & Train

In [24]:
# Cell 13: Train (GPU OPTIMIZED)
import shutil
import os
import sys

# Get the current Python executable path (ensures we use the venv's python)
PYTHON = sys.executable

# Point TensorFlow to system CUDA libraries (vast.ai has CUDA pre-installed)
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/local/cuda'

#=============================================================================
# TRAINING
#=============================================================================
# Clean up previous training run
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)
    print(f"Cleaned up: {MODEL_DIR}")

# Verify GPU is available
print("Checking GPU...")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "nvidia-smi not found"

print()
print("Starting training...")
print(f"  Python: {PYTHON}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Training steps: 20,000")
print(f"  Model: 128 filters (larger)")
print()

# Run training with larger model
!LD_LIBRARY_PATH=/usr/local/cuda/lib64:$LD_LIBRARY_PATH uv run -m microwakeword.model_train_eval \
    --training_config='{config_path}' \
    --train 1 \
    --test_tflite_streaming_quantized 1 \
    --use_weights 'best_weights' \
    mixednet \
    --pointwise_filters '128,128,128,128' \
    --repeat_in_block '1,1,1,1' \
    --mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
    --residual_connection '0,0,0,0' \
    --first_conv_filters 64 \
    --first_conv_kernel_size 5 \
    --stride 3

Cleaned up: /workspace/trained_models
Checking GPU...
Tesla V100-SXM2-32GB, 32768 MiB

Starting training...
  Python: /workspace/.venv/bin/python3
  Batch size: 1024
  Training steps: 20,000
  Model: 128 filters (larger)

2025-11-26 13:30:09.177457: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-26 13:30:09.195548: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-26 13:30:09.201318: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-26 13:30:10.207362: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
INFO:absl:Loading a

## Step 7: Export Model

In [25]:
# Cell 15: Package trained model
import shutil

# Search in the correct directory (MODEL_DIR, not relative path)
model_paths = list(MODEL_DIR.rglob('stream_state_internal_quant.tflite'))

print(f"Searching in: {MODEL_DIR}")
print(f"Found: {model_paths}")

if not model_paths:
    # Also check for non-quantized model
    non_quant_paths = list(MODEL_DIR.rglob('*.tflite'))
    print(f"All TFLite files found: {non_quant_paths}")
    
    if non_quant_paths:
        model_path = non_quant_paths[0]
        output_model = MODEL_DIR / f'{WAKE_WORD_ID}.tflite'
        shutil.copy(model_path, output_model)
        print(f"\nModel: {output_model}")
        print(f"Size: {output_model.stat().st_size:,} bytes")
    else:
        print("ERROR: No TFLite model found!")
else:
    model_path = model_paths[0]
    output_model = MODEL_DIR / f'{WAKE_WORD_ID}.tflite'
    shutil.copy(model_path, output_model)
    print(f"\nModel: {output_model}")
    print(f"Size: {output_model.stat().st_size:,} bytes")

Searching in: /workspace/trained_models
Found: [PosixPath('/workspace/trained_models/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite')]

Model: /workspace/trained_models/hey_daisy.tflite
Size: 132,848 bytes


In [26]:
# Cell 16: Create manifest
manifest = {
    'type': 'micro',
    'wake_word': WAKE_WORD,
    'model': f'{WAKE_WORD_ID}.tflite',
    'version': 1,
    'micro': {'probability_cutoff': 0.5, 'sliding_window_average_size': 10}
}

manifest_path = MODEL_DIR / f'{WAKE_WORD_ID}.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))

{
  "type": "micro",
  "wake_word": "Hey Daisy",
  "model": "hey_daisy.tflite",
  "version": 1,
  "micro": {
    "probability_cutoff": 0.5,
    "sliding_window_average_size": 10
  }
}


In [27]:
# Cell 16: Verify model
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path=str(output_model))
interpreter.allocate_tensors()

inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

print(f"Input: {inp['shape']}, type={inp['dtype']}")
print(f"Output: {out['shape']}, type={out['dtype']}")

if inp['shape'][-1] == 40:
    print("\n✓ Compatible with ESP32 inference code")

Input: [ 1  3 40], type=<class 'numpy.int8'>
Output: [1 1], type=<class 'numpy.uint8'>

✓ Compatible with ESP32 inference code


        See the [migration guide](https://ai.google.dev/edge/litert/migration)
        for details.
        
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## Deployment

1. Copy `daisy.tflite` to SD card at `/models/`
2. Update sketch: `#define MODEL_PATH "/models/daisy.tflite"`

**Tuning:**
- False positives → increase `PROBABILITY_CUTOFF`
- False negatives → decrease `PROBABILITY_CUTOFF`